# 04 · JARVIS voice — GPT-SoVITS TTS engine

**Kernel:** `JARVIS - GPT-SoVITS` (package runtime). **Restart kernel → Run All** for clean VRAM numbers,
and don't run training (notebook 03) at the same time.

- `JarvisVoice.speak(text)` — synthesise in the cloned voice and play. Models stay loaded between calls.
- `JarvisVoice.speak_stream(sentences)` — sentence one plays while sentence two is still synthesising.
  `sentences` can be any iterable, including a generator fed by the LLM.
- Measures time-to-first-audio and peak VRAM with every inference model resident, writes 5 sample WAVs,
  and ends with the 8 GB verdict.

If training hasn't produced weights yet and `ALLOW_ZERO_SHOT` is on, it runs the *pretrained* v2ProPlus model
cloning from the reference clip alone. Latency and VRAM are the same; quality is not. Results are labelled either way.

In [ ]:
import os, sys, json, time, re, queue, threading, statistics
from pathlib import Path
import numpy as np

GSV_ROOT = next(p for p in sorted(Path(r"C:\jarvis-apps").glob("GPT-SoVITS*")) if (p / "webui.py").is_file())
PRE = GSV_ROOT / "GPT_SoVITS" / "pretrained_models"
WORK = Path(r"C:\jarvis-data\voice")
OUT = Path.cwd() / "outputs"          # resolved now: the model-loading cell changes the working directory
OUT.mkdir(exist_ok=True)
EXP_NAME, VERSION = "jarvis_v2pp", "v2ProPlus"
ALLOW_ZERO_SHOT = True
REF_AUDIO, REF_TEXT = None, None      # override the reference picked by notebook 03 (3-10 s clip + its transcript)
SEED = 42
LLM_MB = 6000                         # what the Qwen3-8B LLM is expected to need

SAMPLE_LINES = [
    "Sir, your class begins in thirty minutes.",
    "I'm afraid I can't reach your calendar at the moment, Sir.",
    "Traffic on the usual route is heavy; leaving now puts you there at nine fifty-eight.",
    "You have three unread messages, two of which appear to be from your professor.",
    "Shall I set an alarm for six forty-five tomorrow morning?",
]

def latest(folder, pattern):
    files = sorted(Path(folder).glob(pattern), key=os.path.getmtime)
    return files[-1] if files else None

SOVITS = latest(GSV_ROOT / f"SoVITS_weights_{VERSION}", f"{EXP_NAME}_e*.pth")
GPT = latest(GSV_ROOT / f"GPT_weights_{VERSION}", f"{EXP_NAME}-e*.ckpt")
TRAINED = bool(SOVITS and GPT)
if not TRAINED:
    assert ALLOW_ZERO_SHOT, "no trained weights yet - run notebook 03 or set ALLOW_ZERO_SHOT"
    SOVITS, GPT = PRE / "v2Pro" / "s2Gv2ProPlus.pth", PRE / "s1v3.ckpt"
MODE = "trained" if TRAINED else "zeroshot"

ref = json.loads((WORK / "ref.json").read_text(encoding="utf-8")) if (WORK / "ref.json").exists() else {}
REF_AUDIO = Path(REF_AUDIO or ref.get("path", ""))
REF_TEXT = REF_TEXT or ref.get("text")
assert REF_AUDIO.is_file() and REF_TEXT, "no reference clip: run notebook 03 up to step 3, or set REF_AUDIO/REF_TEXT"
print(f"MODE = {MODE.upper()}\n  SoVITS {SOVITS.name}\n  GPT    {GPT.name}\n  ref    {REF_AUDIO.name}: {REF_TEXT!r}")

In [ ]:
# --- VRAM monitor: create BEFORE anything touches CUDA ---------------------------------
# process MB : this process's dedicated memory on the NVIDIA adapter, from the Windows
#              "GPU Process Memory" counter (what Task Manager shows). NVML cannot see
#              per-process usage under WDDM; this counter can. Headline figure.
# device delta MB : NVML total used minus the baseline. Includes other apps - cross-check.
import ctypes, threading
from ctypes import wintypes
import pynvml

class _FmtValue(ctypes.Structure):
    _fields_ = [("CStatus", wintypes.DWORD), ("largeValue", ctypes.c_longlong)]

class _FmtItem(ctypes.Structure):
    _fields_ = [("szName", wintypes.LPWSTR), ("FmtValue", _FmtValue)]

class PdhWildcard:
    """Every instance of a wildcard Windows performance counter, as {instance: bytes}."""
    def __init__(self, path):
        self.pdh, self.path = ctypes.WinDLL("pdh"), path
        self.query, self.counter = wintypes.HANDLE(), wintypes.HANDLE()
        self.reopen()

    def reopen(self):
        if self.query.value:
            self.pdh.PdhCloseQuery(self.query)
        self.query, self.counter = wintypes.HANDLE(), wintypes.HANDLE()
        if self.pdh.PdhOpenQueryW(None, None, ctypes.byref(self.query)) != 0:
            raise OSError("PdhOpenQueryW failed")
        if self.pdh.PdhAddEnglishCounterW(self.query, self.path, None, ctypes.byref(self.counter)) != 0:
            raise OSError(f"cannot open counter {self.path}")

    def read(self):
        if self.pdh.PdhCollectQueryData(self.query) != 0:
            return {}
        size, count = wintypes.DWORD(0), wintypes.DWORD(0)
        self.pdh.PdhGetFormattedCounterArrayW(self.counter, 0x400, ctypes.byref(size), ctypes.byref(count), None)
        if size.value == 0:
            return {}
        buf = (ctypes.c_byte * size.value)()
        if self.pdh.PdhGetFormattedCounterArrayW(self.counter, 0x400, ctypes.byref(size), ctypes.byref(count), buf) != 0:
            return {}
        items = ctypes.cast(buf, ctypes.POINTER(_FmtItem * count.value)).contents
        return {i.szName: i.FmtValue.largeValue for i in items if i.szName}

class VramMonitor:
    def __init__(self, interval=0.05):
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        self.baseline = pynvml.nvmlDeviceGetMemoryInfo(self.handle).used
        # Hybrid laptop: pick the adapter LUID whose usage matches NVML, i.e. the RTX 4060.
        adapters = PdhWildcard(r"\GPU Adapter Memory(*)\Dedicated Usage").read()
        nvidia = min(adapters, key=lambda name: abs(adapters[name] - self.baseline))
        self.luid = nvidia.split("_phys")[0]
        self.prefix = f"pid_{os.getpid()}_{self.luid}"
        self.procs = PdhWildcard(r"\GPU Process Memory(*)\Dedicated Usage")
        self.peak_proc = self.peak_dev = 0
        self.stages, self._last_reopen = [], time.perf_counter()
        self._lock, self._stop = threading.Lock(), threading.Event()
        threading.Thread(target=self._run, args=(interval,), daemon=True).start()
        print(f"{pynvml.nvmlDeviceGetName(self.handle)} | adapter {self.luid} | "
              f"baseline used by other apps: {self.baseline / 2**20:.0f} MB")

    def _run(self, interval):
        while not self._stop.is_set():
            self.sample()
            time.sleep(interval)

    def sample(self):
        mine = [v for n, v in self.procs.read().items() if n.startswith(self.prefix)]
        if not mine and time.perf_counter() - self._last_reopen > 1:
            self.procs.reopen()  # our instance only appears once CUDA creates a context
            self._last_reopen = time.perf_counter()
        proc = max(mine, default=0)
        dev = pynvml.nvmlDeviceGetMemoryInfo(self.handle).used - self.baseline
        with self._lock:
            self.peak_proc, self.peak_dev = max(self.peak_proc, proc), max(self.peak_dev, dev)
        return proc, dev

    def stage(self, label):
        proc, dev = self.sample()
        entry = {"stage": label, "process_now_mb": round(proc / 2**20), "process_peak_mb": round(self.peak_proc / 2**20),
                 "device_delta_now_mb": round(dev / 2**20), "device_delta_peak_mb": round(self.peak_dev / 2**20)}
        self.stages.append(entry)
        print(f"VRAM [{label}] now {entry['process_now_mb']} MB, peak {entry['process_peak_mb']} MB "
              f"(device delta now {entry['device_delta_now_mb']}, peak {entry['device_delta_peak_mb']})")
        return entry

    def summary(self):
        self.sample()
        return {"peak_process_mb": round(self.peak_proc / 2**20), "peak_device_delta_mb": round(self.peak_dev / 2**20),
                "baseline_other_apps_mb": round(self.baseline / 2**20), "stages": self.stages}

vram = VramMonitor()

In [ ]:
os.chdir(GSV_ROOT)                     # the package resolves model paths relative to its root
for p in ["GPT_SoVITS/eres2net", "GPT_SoVITS/BigVGAN", "tools", "GPT_SoVITS", ""]:
    sys.path.insert(0, str(GSV_ROOT / p))

t = time.perf_counter()
import torch, sounddevice as sd
from scipy.io import wavfile
from GPT_SoVITS.TTS_infer_pack.TTS import TTS
import_ms = (time.perf_counter() - t) * 1000

t = time.perf_counter()
# Version goes inside "custom": TTS_Config lower-cases a top-level version and then rejects "v2proplus".
# The real model version is read from the SoVITS checkpoint.
tts = TTS({"custom": {"device": "cuda", "is_half": True, "version": VERSION,
                      "t2s_weights_path": str(GPT), "vits_weights_path": str(SOVITS),
                      "bert_base_path": str(PRE / "chinese-roberta-wwm-ext-large"),
                      "cnhuhbert_base_path": str(PRE / "chinese-hubert-base")}})
load_ms = (time.perf_counter() - t) * 1000
print(f"torch {torch.__version__} | imports {import_ms:.0f} ms | all models loaded in {load_ms:.0f} ms")
vram.stage("after_load")

In [ ]:
def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]

class JarvisVoice:
    """Cloned JARVIS voice. Models stay loaded; the reference prompt is cached after the first call."""

    def __init__(self, tts, ref_audio, ref_text, seed=SEED):
        self.tts = tts
        self.params = dict(text_lang="en", ref_audio_path=str(ref_audio), prompt_text=ref_text, prompt_lang="en",
                           top_k=15, top_p=1.0, temperature=1.0, text_split_method="cut0", batch_size=1,
                           split_bucket=False, return_fragment=False, speed_factor=1.0, seed=seed,
                           parallel_infer=True, repetition_penalty=1.35)

    def synth(self, text):
        """Returns (sample_rate, int16 mono audio)."""
        sr = audio = None
        for sr, audio in self.tts.run({**self.params, "text": text}):
            pass
        return sr, audio

    def speak(self, text):
        t0 = time.perf_counter()
        sr, audio = self.synth(text)
        synth_ms = (time.perf_counter() - t0) * 1000
        sd.play(audio, sr)
        ttfa_ms = (time.perf_counter() - t0) * 1000
        sd.wait()
        audio_s = len(audio) / sr
        return audio, {"ttfa_ms": round(ttfa_ms), "synth_ms": round(synth_ms), "audio_s": round(audio_s, 2),
                       "rtf": round(synth_ms / 1000 / audio_s, 3)}

    def speak_stream(self, sentences):
        """Synthesise on a worker thread while the main thread plays whatever is ready."""
        ready, events, errors = queue.Queue(maxsize=8), [], []
        t0 = time.perf_counter()
        now = lambda: round((time.perf_counter() - t0) * 1000)

        def producer():
            try:
                for i, sentence in enumerate(sentences):
                    events.append((now(), f"synth {i} start   {sentence[:50]!r}"))
                    sr, audio = self.synth(sentence)
                    events.append((now(), f"synth {i} done    ({len(audio) / sr:.1f} s of audio)"))
                    ready.put((i, sr, audio))
            except Exception as exc:
                errors.append(exc)
            finally:
                ready.put(None)

        threading.Thread(target=producer, daemon=True).start()
        item = ready.get()
        if item is None:
            raise errors[0] if errors else RuntimeError("nothing to say")
        ttfa_ms, waited_ms, audio_s = None, 0, 0.0
        with sd.OutputStream(samplerate=item[1], channels=1, dtype="int16") as stream:
            while item is not None:
                i, sr, audio = item
                ttfa_ms = now() if ttfa_ms is None else ttfa_ms
                events.append((now(), f"play  {i} start"))
                stream.write(audio.reshape(-1, 1))          # blocks at playback pace
                audio_s += len(audio) / sr
                t_wait = time.perf_counter()
                item = ready.get()
                gap = (time.perf_counter() - t_wait) * 1000
                if item is not None and gap > 20:
                    waited_ms += gap
                    events.append((now(), f"GAP   waited {gap:.0f} ms for sentence {item[0]}"))
        events.append((now(), "playback finished"))
        if errors:
            raise errors[0]
        return {"ttfa_ms": ttfa_ms, "total_ms": now(), "audio_s": round(audio_s, 2),
                "gaps_ms": round(waited_ms), "events": events}

voice = JarvisVoice(tts, REF_AUDIO, REF_TEXT)
t = time.perf_counter()
voice.synth("Systems online.")
cold_ms = (time.perf_counter() - t) * 1000
print(f"first synthesis (cold: reference processing + CUDA warm-up): {cold_ms:.0f} ms")
vram.stage("after_first_synth")

## Time-to-first-audio — single sentence

In [ ]:
LINE = SAMPLE_LINES[0]
warm_synth = []
for _ in range(3):
    t = time.perf_counter(); sr, a = voice.synth(LINE); warm_synth.append((time.perf_counter() - t) * 1000)
_, spoken = voice.speak(LINE)
single = {"text": LINE, "cold_first_call_ms": round(cold_ms), "warm_synth_median_ms": round(statistics.median(warm_synth)),
          "warm_synth_runs_ms": [round(x) for x in warm_synth], "spoken": spoken}
print(json.dumps(single, indent=2))
vram.stage("after_single_sentence")

## Streaming — sentence one plays while sentence two synthesises

In [ ]:
PARAGRAPH = ("Good morning, Sir. Your first lecture begins at ten. "
             "Traffic on the usual route is heavy, so leaving by nine fifteen would be wise. Shall I call a cab?")
stream_stats = voice.speak_stream(split_sentences(PARAGRAPH))
for t_ms, event in stream_stats["events"]:
    print(f"{t_ms:7d} ms  {event}")
print(f"\nTTFA {stream_stats['ttfa_ms']} ms | {stream_stats['audio_s']} s spoken in {stream_stats['total_ms']} ms | "
      f"total waiting between sentences {stream_stats['gaps_ms']} ms")
vram.stage("after_stream")

## Samples — ordinary assistant lines, not lines from the source

In [ ]:
# Test renders go to audio/samples/, one folder per model combination, so A/B sets don't overwrite each other.
SAMPLE_DIR = Path(r"P:\Coding\App Development\MyProjects\JARVIS\audio\samples")
if TRAINED:
    epoch = lambda pattern, name: re.search(pattern, name).group(1)
    SAMPLE_DIR /= f"gptsovits_trained_sovits-e{epoch(r'_e(\d+)_s', SOVITS.name)}_gpt-e{epoch(r'-e(\d+)', GPT.name)}"
else:
    SAMPLE_DIR /= "gptsovits_zeroshot_pretrained"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

samples = []
for i, line in enumerate(SAMPLE_LINES, 1):
    t = time.perf_counter()
    sr, audio = voice.synth(line)
    path = SAMPLE_DIR / f"gptsovits_{MODE}_{i}.wav"
    wavfile.write(path, sr, audio)
    samples.append({"file": path.name, "text": line, "synth_ms": round((time.perf_counter() - t) * 1000),
                    "audio_s": round(len(audio) / sr, 2)})
    print(samples[-1])

## VRAM verdict

In [ ]:
vsum = vram.summary()
results = {"mode": MODE, "sovits": SOVITS.name, "gpt": GPT.name, "reference": {"path": str(REF_AUDIO), "text": REF_TEXT},
           "load_ms": round(load_ms), "single_sentence": single,
           "stream": {k: v for k, v in stream_stats.items() if k != "events"}, "samples": samples, "vram": vsum,
           "torch_max_reserved_mb": round(torch.cuda.max_memory_reserved() / 2**20)}
(OUT / f"gptsovits_{MODE}_results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

total_mb = pynvml.nvmlDeviceGetMemoryInfo(vram.handle).total / 2**20
def peak_of(name):
    p = OUT / name
    return json.loads(p.read_text(encoding="utf-8"))["vram"]["peak_process_mb"] if p.exists() else None
whisper_mb, gsv_mb, other_mb = peak_of("whisper_results.json"), vsum["peak_process_mb"], vsum["baseline_other_apps_mb"]

print(f"RTX 4060 total                      {total_mb:7.0f} MB")
print(f"other apps at measurement time      {other_mb:7d} MB")
print(f"Whisper large-v3-turbo int8_float16 {whisper_mb if whisper_mb is not None else 'run notebook 01':>7} MB")
print(f"GPT-SoVITS inference (all models)   {gsv_mb:7d} MB   (torch reserved {results['torch_max_reserved_mb']} MB)")
print(f"LLM budget                          {LLM_MB:7d} MB")
if whisper_mb is not None:
    headroom = total_mb - other_mb - whisper_mb - gsv_mb - LLM_MB
    print(f"headroom                            {headroom:7.0f} MB")
    if headroom >= 300:
        print("\nVERDICT: fits, with room to spare.")
    elif headroom >= 0:
        print("\nVERDICT: fits on paper with under 300 MB spare - any context growth will spill.")
    else:
        print(f"\nVERDICT: does NOT fit - short by {-headroom:.0f} MB. Under WDDM that won't crash: it spills into "
              "shared system RAM and everything gets slow. Options: smaller LLM context/KV cache, Whisper on CPU "
              "(see notebook 01's CPU numbers), or distil this voice into Piper for CPU inference.")